In [5]:
# Import Libraries
# Basic libraries
import pandas as pd
import numpy as np
import re
import string
# NLP libraries
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.tokenize import word_tokenize
# Sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report
# Download required NLTK data
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [3]:
# Load Dataset
df = pd.read_csv("IMDB Dataset.csv")
print("Dataset Shape:", df.shape)
print("\nSample Data:\n", df.head())
print("\nClass Distribution:\n", df['sentiment'].value_counts())

Dataset Shape: (50000, 2)

Sample Data:
                                               review sentiment
0  One of the other reviewers has mentioned that ...  positive
1  A wonderful little production. <br /><br />The...  positive
2  I thought this was a wonderful way to spend ti...  positive
3  Basically there's a family where a little boy ...  negative
4  Petter Mattei's "Love in the Time of Money" is...  positive

Class Distribution:
 sentiment
positive    25000
negative    25000
Name: count, dtype: int64


In [6]:
# NLP Preprocessing
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()
def clean_text(text):
    # Lowercasing
    text = text.lower()
    #Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))
    # Remove numbers
    text = re.sub(r'\d+', '', text)
    # Tokenization
    tokens = word_tokenize(text)
    # Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # Lemmatization (better than stemming for readability)
    tokens = [lemmatizer.lemmatize(word) for word in tokens]
    # Join tokens back
    return " ".join(tokens)
# Apply preprocessing
df['cleaned_text'] = df['review'].apply(clean_text)
print("\nCleaned Sample:\n", df[['review', 'cleaned_text']].head())


Cleaned Sample:
                                               review  \
0  One of the other reviewers has mentioned that ...   
1  A wonderful little production. <br /><br />The...   
2  I thought this was a wonderful way to spend ti...   
3  Basically there's a family where a little boy ...   
4  Petter Mattei's "Love in the Time of Money" is...   

                                        cleaned_text  
0  one reviewer mentioned watching oz episode you...  
1  wonderful little production filming technique ...  
2  thought wonderful way spend time hot summer we...  
3  basically there family little boy jake think t...  
4  petter matteis love time money visually stunni...  


In [7]:
# Convert Labels
df['sentiment'] = df['sentiment'].map({'positive': 1, 'negative': 0})

In [11]:
# Train-Test Split
X = df['cleaned_text']
y = df['sentiment']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
# Feature Engineering
# Bag of Words
bow = CountVectorizer(max_features=5000)
X_train_bow = bow.fit_transform(X_train)
X_test_bow = bow.transform(X_test)
# TF-IDF
tfidf = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [13]:
# Model Training Function
def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    print("Model:", model.__class__.__name__)
    print("Accuracy:", accuracy_score(y_test, y_pred))
    print("Precision:", precision_score(y_test, y_pred))
    print("Recall:", recall_score(y_test, y_pred))
    print("F1 Score:", f1_score(y_test, y_pred))
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("-" * 50)

In [14]:
print(" Using Bag of Words ")
evaluate_model(LogisticRegression(), X_train_bow, X_test_bow, y_train, y_test)
evaluate_model(MultinomialNB(), X_train_bow, X_test_bow, y_train, y_test)
evaluate_model(DecisionTreeClassifier(), X_train_bow, X_test_bow, y_train, y_test)

----- Using Bag of Words -----


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Model: LogisticRegression
Accuracy: 0.8732
Precision: 0.8697783879191998
Recall: 0.8801349474102005
F1 Score: 0.8749260209114224

Classification Report:
               precision    recall  f1-score   support

           0       0.88      0.87      0.87      4961
           1       0.87      0.88      0.87      5039

    accuracy                           0.87     10000
   macro avg       0.87      0.87      0.87     10000
weighted avg       0.87      0.87      0.87     10000

--------------------------------------------------
Model: MultinomialNB
Accuracy: 0.8447
Precision: 0.8495788206979543
Recall: 0.8406429847191903
F1 Score: 0.8450872817955112

Classification Report:
               precision    recall  f1-score   support

           0       0.84      0.85      0.84      4961
           1       0.85      0.84      0.85      5039

    accuracy                           0.84     10000
   macro avg       0.84      0.84      0.84     10000
weighted avg       0.84      0.84      0.84    

In [15]:
print(" Using TF-IDF")
evaluate_model(LogisticRegression(), X_train_tfidf, X_test_tfidf, y_train, y_test)
evaluate_model(MultinomialNB(), X_train_tfidf, X_test_tfidf, y_train, y_test)
evaluate_model(DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf, y_train, y_test)

 Using TF-IDF
Model: LogisticRegression
Accuracy: 0.8849
Precision: 0.8764523625096824
Recall: 0.8981940861282001
F1 Score: 0.8871900421444673

Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.87      0.88      4961
           1       0.88      0.90      0.89      5039

    accuracy                           0.88     10000
   macro avg       0.89      0.88      0.88     10000
weighted avg       0.89      0.88      0.88     10000

--------------------------------------------------
Model: MultinomialNB
Accuracy: 0.849
Precision: 0.847274158630191
Recall: 0.8543361778130582
F1 Score: 0.8507905138339921

Classification Report:
               precision    recall  f1-score   support

           0       0.85      0.84      0.85      4961
           1       0.85      0.85      0.85      5039

    accuracy                           0.85     10000
   macro avg       0.85      0.85      0.85     10000
weighted avg       0.85      0.85  

In [16]:
results = []
def store_results(name, model, X_train, X_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1 Score": f1_score(y_test, y_pred)
    })
store_results("Logistic Regression (TF-IDF)", LogisticRegression(), X_train_tfidf, X_test_tfidf)
store_results("Naive Bayes (TF-IDF)", MultinomialNB(), X_train_tfidf, X_test_tfidf)
store_results("Decision Tree (TF-IDF)", DecisionTreeClassifier(), X_train_tfidf, X_test_tfidf)
results_df = pd.DataFrame(results)
print(results_df)

                          Model  Accuracy  Precision    Recall  F1 Score
0  Logistic Regression (TF-IDF)    0.8849   0.876452  0.898194  0.887190
1          Naive Bayes (TF-IDF)    0.8490   0.847274  0.854336  0.850791
2        Decision Tree (TF-IDF)    0.7137   0.718825  0.709268  0.714015
